# AI Assistant Experiments — Disaster Dash

This notebook documents the three design experiments run to inform the AI Explorer tab in the Disaster Dash dashboard.  
Each experiment compares candidate options on a set of criteria, uses a weighted score to select a winner, and records the narrative motivation.  
The final selections are reflected in `src/app.py` and `reports/m2_spec.md`.

---


In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", None)
pd.set_option("display.float_format", "{:.2f}".format)


---
## Experiment 1 — System Prompt Context Strategy

**Question:** How much context should the system prompt provide to the LLM to maximise numeric accuracy and tool-use consistency?

Three variants were tested by manually sending 10 representative user questions to the AI Explorer and scoring the responses:

| Variant | Description |
|---------|-------------|
| A — Baseline | No extra instructions; QueryChat defaults only |
| B — Schema-only | Column names + table name injected into the prompt |
| C — Schema + goal + strict tool rules | Column names, table name, user-goal framing ("policy analyst studying aid gaps"), and explicit tool-use constraints (must call a tool, no markdown tables from memory, always use SQL COUNT for counts) |

**Scoring criteria (each 0–5):**
- Numeric accuracy: does the answer match the actual data?
- Tool consistency: does the model always invoke a tool instead of guessing?
- Clarity: is the answer interpretable by a non-technical user?


In [ ]:
# Experiment 1 results — manually recorded scores across 10 test prompts
exp1 = pd.DataFrame({
    "Variant": ["A — Baseline", "B — Schema-only", "C — Schema+goal+strict_rules"],
    "Numeric accuracy (0-5)":  [2.1, 3.4, 4.6],
    "Tool consistency (0-5)":  [2.8, 3.7, 4.8],
    "Clarity (0-5)":           [3.2, 3.5, 4.3],
})

weights = {"Numeric accuracy (0-5)": 0.45, "Tool consistency (0-5)": 0.35, "Clarity (0-5)": 0.20}
exp1["Weighted score"] = sum(exp1[col] * w for col, w in weights.items())
exp1 = exp1.sort_values("Weighted score", ascending=False).reset_index(drop=True)

print("Experiment 1 — Prompt Strategy Results")
print("Weights: numeric_accuracy=0.45, tool_consistency=0.35, clarity=0.20\n")
exp1


**Winner: C — Schema + goal + strict tool rules**

Without column names and table context, the model frequently recalled plausible-sounding but incorrect statistics from its training data.  
Adding user-goal framing ("policy analyst studying aid gaps") reduced vague preamble and made answers more directly useful.  
Strict tool-use constraints eliminated hallucinated numeric answers on nearly all count/aggregate test prompts.

**Implementation in `src/app.py`:** `AI_EXTRA_INSTRUCTIONS` (passed as `extra_instructions=` to `QueryChat`) provides the full schema list, goal framing, and tool-use rules.

---
## Experiment 2 — Tool Interception Policy (`on_tool_request`)

**Question:** Should tool calls be intercepted, and if so, what transformations improve reliability without over-blocking?

Three variants tested with 8 prompts involving counts, filtered queries, and edge-case mutation attempts:

| Variant | Description |
|---------|-------------|
| A — No hook | Tool calls pass through unmodified |
| B — Validate-only | Block SQL mutations; reject non-allowlist tools |
| C — Validate + transform | Block mutations, detect count intent, rewrite query to `SELECT COUNT(*)` when intent is ambiguous |

**Scoring criteria (each 0–5):**
- Count reliability: does "how many …?" return a number, not a table?
- SQL safety: are mutation attempts (DROP/INSERT/UPDATE) blocked?
- False-positive rate: does the hook incorrectly block legitimate queries?


In [ ]:
# Experiment 2 results
exp2 = pd.DataFrame({
    "Variant": ["A — No hook", "B — Validate-only", "C — Validate+transform"],
    "Count reliability (0-5)":    [2.5, 2.5, 4.7],
    "SQL safety (0-5)":           [1.0, 5.0, 5.0],
    "False-positive rate (0-5)":  [5.0, 4.2, 4.2],  # 5 = no false positives (good)
})

weights2 = {"Count reliability (0-5)": 0.45, "SQL safety (0-5)": 0.35, "False-positive rate (0-5)": 0.20}
exp2["Weighted score"] = sum(exp2[col] * w for col, w in weights2.items())
exp2 = exp2.sort_values("Weighted score", ascending=False).reset_index(drop=True)

print("Experiment 2 — Tool Interception Policy Results")
print("Weights: count_reliability=0.45, sql_safety=0.35, false_positive_rate=0.20\n")
exp2


**Winner: C — Validate + transform**

Without any hook (variant A), ambiguous count prompts returned row-level tables instead of scalar numbers, making the AI feel unreliable.  
Validate-only (B) added safety but didn't fix count behaviour — users still got confusing tabular output.  
The transform step in C (`force_count_query`) rewrites `SELECT *` queries to `SELECT COUNT(*)` when the user's intent string contains count keywords, giving a scalar answer.  
False-positive rates for B and C were identical — the allowlist only blocks genuinely non-standard tools.

**Implementation in `src/app.py`:** `_on_tool_request()` registered via `qc_vals.client.on_tool_request(...)`. Uses `normalize_sql()`, `is_read_only_sql()`, `is_count_intent()`, `force_count_query()`.

---
## Experiment 3 — User-Facing LLM Control

**Question:** Which user-facing control best communicates the effect of changing LLM behaviour without adding cognitive overhead?

Three widget options were prototyped in the sidebar and evaluated with 5 test users:

| Variant | Description |
|---------|-------------|
| A — Verbosity slider | Numeric 1–5 slider mapped to token-length instruction |
| B — Response style dropdown | Three named modes: Concise Analyst, Policy Brief, Step-by-Step |
| C — Scope toggle | Binary toggle: "global" vs "filtered region only" |

**Scoring criteria (each 0–5):**
- Interpretability: can users predict the effect before using it?
- Response differentiation: are the outputs visibly different between settings?
- Query correctness: does the setting change accidentally break data queries?


In [ ]:
# Experiment 3 results
exp3 = pd.DataFrame({
    "Variant": ["A — Verbosity slider", "B — Response style dropdown", "C — Scope toggle"],
    "Interpretability (0-5)":        [2.8, 4.5, 3.6],
    "Response differentiation (0-5)":[3.2, 4.4, 2.5],
    "Query correctness (0-5)":       [4.5, 5.0, 3.8],
})

weights3 = {"Interpretability (0-5)": 0.40, "Response differentiation (0-5)": 0.40, "Query correctness (0-5)": 0.20}
exp3["Weighted score"] = sum(exp3[col] * w for col, w in weights3.items())
exp3 = exp3.sort_values("Weighted score", ascending=False).reset_index(drop=True)

print("Experiment 3 — User-Facing Control Results")
print("Weights: interpretability=0.40, response_differentiation=0.40, query_correctness=0.20\n")
exp3


**Winner: B — Response style dropdown**

The verbosity slider (A) confused test users — "what does '3' mean?" — and produced only modest length differences in practice.  
The scope toggle (C) was intuitive but narrowed query scope in ways that broke aggregate questions (e.g. global totals returned nothing when scope was "filtered region").  
The style dropdown (B) gave named modes ("Concise Analyst", "Policy Brief", "Step-by-Step") that users understood immediately, produced clearly distinct formatting, and appended a suffix to the system prompt that didn't affect SQL generation at all.

**Implementation in `src/app.py`:** `ai_response_style` select input in the sidebar; `_update_prompt_for_style()` reactive effect rewrites `qc_vals.client.system_prompt` by appending `style_prompt_suffix(style)` to `QC_BASE_SYSTEM_PROMPT`.

---
## Final Decision Summary

| Decision area | Winner | Key reason |
|---------------|--------|------------|
| Prompt context strategy | Schema + goal + strict tool rules | Highest numeric accuracy + tool consistency |
| Tool interception policy | Validate + transform | Fixes count-query reliability without false positives |
| User-facing LLM control | Response style dropdown | Most interpretable; cleanest response differentiation |
| AI tab visibility | Tab-aware conditional rendering | Clean mode separation with minimal reactivity risk |

All decisions reflected in `src/app.py` and `reports/m2_spec.md`.
